# Run a classifer on the branchwater search output

In [1]:
# base checkout of workflow directory is here:
BASE='/home/ctbrown/scratch3/2025-workflow-core99/'

# parquet files from branchwater
#BASE_OUTPUTS=BASE+'/outputs.branchwater'

MIN=90
SCALED=1000
HASH_THRESHOLD=int(20000 / SCALED)

CORENAMES = set([ x.strip() for x in open(BASE + 'inputs.cds/names.list') ])
print(len(CORENAMES))

#BASE_OUTPUTS=BASE+f'/outputs.mapping/cds/singlehash.k21/outputs.branchwater.min{MIN}.scaled{SCALED}'
BASE_OUTPUTS=BASE+'/outputs.cds/branchwater.cds3'

16


In [2]:
import polars as pl
import numpy as np
import sklearn.tree
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold

In [3]:
dirpath = BASE_OUTPUTS+'/*.parquet'

bw_df = pl.scan_parquet(dirpath).collect()
bw_df = bw_df.with_columns(pl.col("query_name").alias("species"),
                           pl.col("match_name").alias("acc"))
bw_df = bw_df.select(["species", "acc", "containment", "intersect_hashes"])

# mimic the Web site search threshold of 0.1
bw_df = bw_df.filter(pl.col("intersect_hashes") >= HASH_THRESHOLD)
bw_df = bw_df.filter(pl.col("species").is_in(CORENAMES))
bw_df

species,acc,containment,intersect_hashes
str,str,f64,i64
"""s__Bariatricus sp004560705""","""ERR2241674""",0.504614,1531
"""s__Bariatricus sp004560705""","""ERR2241669""",0.504614,1531
"""s__Bariatricus sp004560705""","""SRR11185229""",0.488134,1481
"""s__Bariatricus sp004560705""","""SRR9866666""",0.479235,1454
"""s__Bariatricus sp004560705""","""SRR11185232""",0.478906,1453
…,…,…,…
"""s__UBA2868 sp004552595""","""SRR5919199""",0.003175,20
"""s__UBA2868 sp004552595""","""SRR5788024""",0.003175,20
"""s__UBA2868 sp004552595""","""SRR26115360""",0.003175,20


In [4]:
bw_df['species'].n_unique()

16

In [5]:
metadata_df = pl.scan_parquet('/group/ctbrowngrp5/sra-metagenomes/20241128-metadata.parquet')
metadata_df = metadata_df.select(["acc", "organism", "assay_type"])\
    .filter(pl.col("acc") != "NP")\
    .filter(pl.col("assay_type") == "WGS")

metadata_df = metadata_df.collect()
metadata_df

acc,organism,assay_type
str,str,str
"""SRR28523869""","""human metagenome""","""WGS"""
"""SRR19901137""","""Streptococcus suis""","""WGS"""
"""SRR24962475""","""Escherichia coli""","""WGS"""
"""SRR29161330""","""biofilm metagenome""","""WGS"""
"""SRR26668739""","""bovine gut metagenome""","""WGS"""
…,…,…
"""ERR4174288""","""human gut metagenome""","""WGS"""
"""ERR3160109""","""human gut metagenome""","""WGS"""
"""ERR2709457""","""human gut metagenome""","""WGS"""


In [6]:
bw_df = bw_df.join(metadata_df, on="acc", how="left")
bw_df

species,acc,containment,intersect_hashes,organism,assay_type
str,str,f64,i64,str,str
"""s__Bariatricus sp004560705""","""ERR2241674""",0.504614,1531,"""metagenome""","""WGS"""
"""s__Bariatricus sp004560705""","""ERR2241669""",0.504614,1531,"""metagenome""","""WGS"""
"""s__Bariatricus sp004560705""","""SRR11185229""",0.488134,1481,"""pig gut metagenome""","""WGS"""
"""s__Bariatricus sp004560705""","""SRR9866666""",0.479235,1454,"""Lawsonia intracellularis""","""WGS"""
"""s__Bariatricus sp004560705""","""SRR11185232""",0.478906,1453,"""pig gut metagenome""","""WGS"""
…,…,…,…,…,…
"""s__UBA2868 sp004552595""","""SRR5919199""",0.003175,20,"""soil metagenome""","""WGS"""
"""s__UBA2868 sp004552595""","""SRR5788024""",0.003175,20,"""marine metagenome""","""WGS"""
"""s__UBA2868 sp004552595""","""SRR26115360""",0.003175,20,"""root metagenome""","""WGS"""


In [7]:
bw_df['organism'].value_counts().sort(by='count', descending=True)

organism,count
str,u32
"""pig gut metagenome""",89038
null,55105
"""human gut metagenome""",39314
"""gut metagenome""",39142
"""soil metagenome""",32402
…,…
"""saltern metagenome""",1
"""Cryptotympana facialis""",1
"""Enterococcus lacertideformus""",1


In [8]:
rename_df = []
for organism in bw_df['organism'].unique().to_list():
    simpleorg = 'unknown'
    if organism:
        for kw in ['human', 'homo']:
            if kw in organism.lower():
                simpleorg = 'human'
                break
        for kw in ['pig', 'sus', 'scrofa']:
            if kw in organism.lower():
                simpleorg = 'pig'
                break
    rename_df.append(dict(organism=organism, simpleorg=simpleorg))

rename_df = pl.DataFrame(rename_df)
rename_df['simpleorg'].value_counts().sort(by='count', descending=True)


simpleorg,count
str,u32
"""unknown""",518
"""human""",20
"""pig""",9


In [9]:
rename_df

organism,simpleorg
str,str
"""Lactobacillus sp.""","""unknown"""
"""human urinary tract metagenome""","""human"""
"""Leclercia adecarboxylata""","""unknown"""
"""Phormidium yuhuli AB48""","""unknown"""
"""Neotoma albigula""","""unknown"""
…,…
"""Corynebacterium simulans""","""unknown"""
"""Cancris carmenensis""","""unknown"""
"""Cervus nippon""","""unknown"""


In [10]:
# add a new column with a simplified organism
bw_df = bw_df.join(rename_df, on='organism', how='inner', coalesce=True).select(['acc', 'organism', 'species', 'simpleorg'])
bw_df.select(['acc', 'simpleorg']).unique()['simpleorg'].value_counts()


simpleorg,count
str,u32
"""unknown""",60384
"""pig""",7369
"""human""",29295


In [11]:
bw_df

acc,organism,species,simpleorg
str,str,str,str
"""ERR2241674""","""metagenome""","""s__Bariatricus sp004560705""","""unknown"""
"""ERR2241669""","""metagenome""","""s__Bariatricus sp004560705""","""unknown"""
"""SRR11185229""","""pig gut metagenome""","""s__Bariatricus sp004560705""","""pig"""
"""SRR9866666""","""Lawsonia intracellularis""","""s__Bariatricus sp004560705""","""unknown"""
"""SRR11185232""","""pig gut metagenome""","""s__Bariatricus sp004560705""","""pig"""
…,…,…,…
"""SRR5919199""","""soil metagenome""","""s__UBA2868 sp004552595""","""unknown"""
"""SRR5788024""","""marine metagenome""","""s__UBA2868 sp004552595""","""unknown"""
"""SRR26115360""","""root metagenome""","""s__UBA2868 sp004552595""","""unknown"""


In [12]:
def make_matrices(df):
    acc_df = df['acc'].unique().to_frame().with_row_index(name='acc_index')
    species_df = df['species'].unique().to_frame().with_row_index(name='species_index')
    org_df = df['simpleorg'].unique().to_frame().with_row_index(name='org_index')

    df = df.join(org_df, on='simpleorg', how='left')
    df = df.join(acc_df, on='acc', how='left')
    df = df.join(species_df, on='species', how='left')

    obs = np.zeros((len(acc_df), len(species_df)))
    target = np.zeros((len(acc_df)))

    for row in df.iter_rows(named=True):
        acc_id = row["acc_index"]
        org_id = row["org_index"]
        species_id = row["species_index"]

        obs[acc_id, species_id] = 1
        target[acc_id] = org_id

    print(f'observations matrix shape is: {obs.shape}')

    return df, obs, target


In [13]:
human_pig_only = bw_df.filter(pl.col("simpleorg") != "unknown")
hp_df, hp_obs, hp_target = make_matrices(human_pig_only)

observations matrix shape is: (36664, 16)


In [14]:
hp_df

acc,organism,species,simpleorg,org_index,acc_index,species_index
str,str,str,str,u32,u32,u32
"""SRR11185229""","""pig gut metagenome""","""s__Bariatricus sp004560705""","""pig""",1,31175,12
"""SRR11185232""","""pig gut metagenome""","""s__Bariatricus sp004560705""","""pig""",1,3482,12
"""SRR8960383""","""pig gut metagenome""","""s__Bariatricus sp004560705""","""pig""",1,20408,12
"""SRR17241623""","""pig gut metagenome""","""s__Bariatricus sp004560705""","""pig""",1,31106,12
"""SRR11185262""","""pig gut metagenome""","""s__Bariatricus sp004560705""","""pig""",1,29064,12
…,…,…,…,…,…,…
"""SRR25939073""","""human blood metagenome""","""s__UBA2868 sp004552595""","""human""",0,24630,13
"""SRR25132618""","""human lung metagenome""","""s__UBA2868 sp004552595""","""human""",0,5619,13
"""SRR14369181""","""pig gut metagenome""","""s__UBA2868 sp004552595""","""pig""",1,28539,13


## 6-Fold Cross Validation approach

Run 6 different splits of the data; train on 1/6 of the data and test on the other 5/6s of the data, and then repeat that for each of the 6 splits.

In [15]:
kf = StratifiedKFold(n_splits=6)

accuracies = []
i = 1
for train_sub, test_sub in kf.split(hp_obs, hp_target):
    dt = DecisionTreeClassifier(random_state=42)
    tree = dt.fit(hp_obs[train_sub], hp_target[train_sub])
    pred = tree.predict(hp_obs[test_sub])

    accuracy = balanced_accuracy_score(hp_target[test_sub], pred)
    accuracies.append(accuracy)
    print(f"iteration {i}: accuracy {accuracy:.3f}")
    i += 1

print(f"mean accuracy across {i-1} splits: {np.mean(accuracies):.3f}")

iteration 1: accuracy 0.968
iteration 2: accuracy 0.974
iteration 3: accuracy 0.975
iteration 4: accuracy 0.967
iteration 5: accuracy 0.970
iteration 6: accuracy 0.973
mean accuracy across 6 splits: 0.971


## What do we after 6-fold cross validation?

This approach lets us check to see that, in general, the performance of any classifer trained on this data will be pretty good - independent of which set of data we use.

Now, to train the best possible classifier for _future_ use, we will use all the data to train:

In [16]:
dt = DecisionTreeClassifier(random_state=42)
tree = dt.fit(hp_obs, hp_target)
pred = tree.predict(hp_obs)

accuracy = balanced_accuracy_score(hp_target, pred)
print(f"full classifier: accuracy {accuracy:.3f}")

full classifier: accuracy 0.976


## What do we do with the full classifier now??

Now we can go back to the original data - not just the human/pig subset, but the one with unknowns. (Or, potentially, new data sets.)

The only thing we need to be careful about is to encode new observations using the species indexes that match the trained classifier, as well as the organism indexes (0 pig, 1 human). Let's extract those:

In [17]:
species_index_df = hp_df.select(['species_index', 'species']).unique()
species_index_df

species_index,species
u32,str
11,"""s__JAFBIX01 sp021531895"""
1,"""s__Cryptobacteroides sp0340892…"
15,"""s__Sodaliphilus sp004557565"""
9,"""s__Cryptobacteroides sp0004326…"
10,"""s__Ornithospirochaeta sp022785…"
…,…
14,"""s__Mogibacterium_A kristiansen…"
2,"""s__Colivicinus sp002299675"""
6,"""s__Floccifex porci"""


In [18]:
org_df = hp_df.select(['org_index', 'simpleorg']).unique()
org_df

org_index,simpleorg
u32,str
1,"""pig"""
0,"""human"""


Now, let's take some of some of the data we didn't touch from the original bw_df, and encode and classify it: 

In [19]:
def make_obs_matrix(df, species_df):
    # need a new access mapping
    acc_df = df['acc'].unique().to_frame().with_row_index(name='acc_index')

    # construct new internal data frame, populate
    df = df.join(acc_df, on='acc', how='left')
    df = df.join(species_df, on='species', how='left')

    obs = np.zeros((len(acc_df), len(species_df)))

    for row in df.iter_rows(named=True):
        acc_id = row["acc_index"]
        species_id = row["species_index"]

        obs[acc_id, species_id] = 1

    print(f'observations matrix shape is: {obs.shape}')

    return df, obs


In [20]:
unknown_only_df = bw_df.filter(pl.col("simpleorg") == "unknown")
unk_df, unk_obs = make_obs_matrix(unknown_only_df, species_index_df)

observations matrix shape is: (60384, 16)


In [21]:
unknown_pred = tree.predict(unk_obs)
unknown_pred

array([0., 0., 0., ..., 0., 0., 0.], shape=(60384,))

## Predictions are great and all but...

Great! XXX predictions 😭 . What do we do with them??

Let's merge the predictions back into the spreadsheet.

In [22]:
unk_pred_df = []
for (row_n, predicted_org) in enumerate(unknown_pred):
    unk_pred_df.append(dict(acc_index=row_n, org_index=int(predicted_org)))
unk_pred_df = pl.DataFrame(unk_pred_df).join(org_df, on='org_index', how='left')

unk_acc_df = unk_df.select(["acc", "acc_index"]).unique()

unk_pred_df = unk_pred_df.join(unk_acc_df, on='acc_index', how='left')
unk_pred_df

acc_index,org_index,simpleorg,acc
i64,i64,str,str
0,0,"""human""","""SRR26638145"""
1,0,"""human""","""SRR2232276"""
2,0,"""human""","""ERR878247"""
3,0,"""human""","""SRR14932690"""
4,0,"""human""","""SRR17658291"""
…,…,…,…
60379,0,"""human""","""SRR25158266"""
60380,0,"""human""","""SRR27603317"""
60381,0,"""human""","""ERR4682886"""


In [23]:
unk_pred_df["simpleorg"].value_counts().sort(by='count', descending=True)

simpleorg,count
str,u32
"""human""",49828
"""pig""",10556


## OK! We have our predictions... did this actually work?

In [24]:
# sample 5 randomly
unk_pred_df.sample(n=5)

acc_index,org_index,simpleorg,acc
i64,i64,str,str
30866,0,"""human""","""SRR29888958"""
20348,0,"""human""","""SRR16350207"""
19991,1,"""pig""","""SRR3990173"""
18032,1,"""pig""","""SRR8931195"""
33968,0,"""human""","""SRR17256064"""


In [25]:
# sample 5 predicted as pig
unk_pred_df.filter(pl.col("simpleorg") == "pig").sample(n=5)

acc_index,org_index,simpleorg,acc
i64,i64,str,str
37598,1,"""pig""","""SRR23311176"""
59432,1,"""pig""","""SRR25584962"""
338,1,"""pig""","""SRR28982154"""
43108,1,"""pig""","""SRR9866660"""
17222,1,"""pig""","""SRR25508143"""


In [26]:
# sample 5 predicted as human
unk_pred_df.filter(pl.col("simpleorg") == "human").sample(n=5)

acc_index,org_index,simpleorg,acc
i64,i64,str,str
32918,0,"""human""","""SRR12315382"""
7972,0,"""human""","""SRR13742830"""
14010,0,"""human""","""SRR2228635"""
5561,0,"""human""","""SRR5788219"""
331,0,"""human""","""SRR6350024"""


# What features are most important for good classification?

First, let's get the feature importances.

In [27]:
importances_df = []
for (species_index, importance) in enumerate(tree.feature_importances_):
    d = dict(species_index=species_index, importance=importance)
    importances_df.append(d)

importances_df = pl.DataFrame(importances_df)
importances_df = importances_df.join(species_index_df, on='species_index', how='left')
importances_df = importances_df.sort(by='importance', descending=True)
importances_df

species_index,importance,species
i64,f64,str
1,0.84157,"""s__Cryptobacteroides sp0340892…"
13,0.063971,"""s__UBA2868 sp004552595"""
10,0.016082,"""s__Ornithospirochaeta sp022785…"
3,0.014611,"""s__Fimisoma sp002320005"""
0,0.013582,"""s__Lactobacillus amylovorus"""
…,…,…
14,0.003513,"""s__Mogibacterium_A kristiansen…"
15,0.002805,"""s__Sodaliphilus sp004557565"""
2,0.002392,"""s__Colivicinus sp002299675"""


Now let's rebuild the classifier after removing each species, and see how well it does...

In [28]:
species_in_order = importances_df['species'].to_list()
species_in_order

['s__Cryptobacteroides sp034089285',
 's__UBA2868 sp004552595',
 's__Ornithospirochaeta sp022785155',
 's__Fimisoma sp002320005',
 's__Lactobacillus amylovorus',
 's__Bariatricus sp004560705',
 's__Prevotella sp002251295',
 's__Cryptobacteroides sp000434935',
 's__JAFBIX01 sp021531895',
 's__Floccifex porci',
 's__Cryptobacteroides sp900546925',
 's__Mogibacterium_A kristiansenii',
 's__Sodaliphilus sp004557565',
 's__Colivicinus sp002299675',
 's__Prevotella sp000434975',
 's__Cryptobacteroides sp000432655']

In [29]:
human_pig_only = bw_df.filter(pl.col("simpleorg") != "unknown")
species_to_rm = list(species_in_order)

remove_df = human_pig_only
while len(species_to_rm) >= 2:
    rm_species = species_to_rm.pop(0)
    print(f'removing {rm_species} too')
    if len(species_to_rm) == 1:
        print('left:', species_to_rm[0])

    remove_df = remove_df.filter(pl.col("species") != rm_species)
    augmented_df, rm_obs, rm_target = make_matrices(remove_df)

    dt = DecisionTreeClassifier(random_state=42)
    tree = dt.fit(rm_obs, rm_target)
    pred = tree.predict(rm_obs)

    accuracy = balanced_accuracy_score(rm_target, pred)
    print(f"classifier accuracy with {remove_df['species'].n_unique()} is: {accuracy:.3f}")
    print('---')

removing s__Cryptobacteroides sp034089285 too
observations matrix shape is: (36660, 15)
classifier accuracy with 15 is: 0.975
---
removing s__UBA2868 sp004552595 too
observations matrix shape is: (36528, 14)
classifier accuracy with 14 is: 0.974
---
removing s__Ornithospirochaeta sp022785155 too
observations matrix shape is: (36525, 13)
classifier accuracy with 13 is: 0.969
---
removing s__Fimisoma sp002320005 too
observations matrix shape is: (35652, 12)
classifier accuracy with 12 is: 0.971
---
removing s__Lactobacillus amylovorus too
observations matrix shape is: (33983, 11)
classifier accuracy with 11 is: 0.971
---
removing s__Bariatricus sp004560705 too
observations matrix shape is: (33927, 10)
classifier accuracy with 10 is: 0.967
---
removing s__Prevotella sp002251295 too
observations matrix shape is: (33820, 9)
classifier accuracy with 9 is: 0.956
---
removing s__Cryptobacteroides sp000434935 too
observations matrix shape is: (31676, 8)
classifier accuracy with 8 is: 0.955
---


## Remove in reverse order...

In [30]:
human_pig_only = bw_df.filter(pl.col("simpleorg") != "unknown")
species_to_rm = list(reversed(species_in_order))

remove_df = human_pig_only
while len(species_to_rm) >= 2:
    remove_df = remove_df.filter(pl.col("species") != rm_species)
    augmented_df, rm_obs, rm_target = make_matrices(remove_df)

    dt = DecisionTreeClassifier(random_state=42)
    tree = dt.fit(rm_obs, rm_target)
    pred = tree.predict(rm_obs)

    accuracy = balanced_accuracy_score(rm_target, pred)
    print(f"classifier accuracy with {remove_df['species'].n_unique()} is: {accuracy:.3f}")
    print('---')

    rm_species = species_to_rm.pop(0)
    print(f'removing {rm_species} too')

observations matrix shape is: (29364, 15)
classifier accuracy with 15 is: 0.974
---
removing s__Cryptobacteroides sp000432655 too
observations matrix shape is: (29139, 14)
classifier accuracy with 14 is: 0.973
---
removing s__Prevotella sp000434975 too
observations matrix shape is: (29139, 14)
classifier accuracy with 14 is: 0.973
---
removing s__Colivicinus sp002299675 too
observations matrix shape is: (29069, 13)
classifier accuracy with 13 is: 0.973
---
removing s__Sodaliphilus sp004557565 too
observations matrix shape is: (26304, 12)
classifier accuracy with 12 is: 0.974
---
removing s__Mogibacterium_A kristiansenii too
observations matrix shape is: (19211, 11)
classifier accuracy with 11 is: 0.971
---
removing s__Cryptobacteroides sp900546925 too
observations matrix shape is: (18480, 10)
classifier accuracy with 10 is: 0.970
---
removing s__Floccifex porci too
observations matrix shape is: (18475, 9)
classifier accuracy with 9 is: 0.969
---
removing s__JAFBIX01 sp021531895 too
obs

## Try something different



In [31]:
species_in_order = importances_df['species'].to_list()
species_in_order

['s__Cryptobacteroides sp034089285',
 's__UBA2868 sp004552595',
 's__Ornithospirochaeta sp022785155',
 's__Fimisoma sp002320005',
 's__Lactobacillus amylovorus',
 's__Bariatricus sp004560705',
 's__Prevotella sp002251295',
 's__Cryptobacteroides sp000434935',
 's__JAFBIX01 sp021531895',
 's__Floccifex porci',
 's__Cryptobacteroides sp900546925',
 's__Mogibacterium_A kristiansenii',
 's__Sodaliphilus sp004557565',
 's__Colivicinus sp002299675',
 's__Prevotella sp000434975',
 's__Cryptobacteroides sp000432655']

In [32]:
human_pig_only = bw_df.filter(pl.col("simpleorg") != "unknown")
species_to_use = list(species_in_order)

for i in range(1, 3): #len(species_to_use)):
    keep_species = species_to_use[:i]
    print(f'using {len(keep_species)} species.')

    class_df = human_pig_only.filter(pl.col("species").is_in(keep_species))
    print(class_df['simpleorg'].value_counts())
    augmented_df, keep_obs, keep_target = make_matrices(class_df)

    dt = DecisionTreeClassifier(random_state=42)
    tree = dt.fit(keep_obs, keep_target)
    pred = tree.predict(keep_obs)

    accuracy = balanced_accuracy_score(keep_target, pred)
    print(f"classifier accuracy with {class_df['species'].n_unique()} is: {accuracy:.3f}")

    # NOW, apply the trained classifier to full set of observations (for whatever list of
    # species we are using).

    # pull out the species mapping.
    species_index_df = augmented_df.select(['species_index', 'species']).unique()

    # filter the branchwater dataframe to contain just the relevant species
    obs_df = bw_df.filter(pl.col("species").is_in(keep_species))
    print(len(obs_df))
    new_df, new_obs = make_obs_matrix(obs_df, species_index_df)

    # run the prediction
    new_pred = tree.predict(new_obs)
    
    org_df = augmented_df.select(['org_index', 'simpleorg']).unique()

    # convert predictions into dataframe
    new_pred_df = []
    for (row_n, predicted_org) in enumerate(new_pred):
        new_pred_df.append(dict(acc_index=row_n, org_index=int(predicted_org)))

    new_pred_df = pl.DataFrame(new_pred_df)
    print('new_pred_df:', new_pred_df)
        
    new_pred_df = new_pred_df.join(org_df, on='org_index', how='inner').with_columns(
        pl.col("simpleorg").alias("predicted_org")
    ).select(['acc_index', 'predicted_org'])
    print(new_pred_df)

    aug_df = new_df.select(['acc_index', 'simpleorg']).unique()
    print('aug_df:', aug_df)
    new_pred_df = new_pred_df.join(aug_df, on='acc_index', how='inner')

    print(new_pred_df.filter(pl.col('predicted_org') == 'pig')['simpleorg'].value_counts())
    #print(new_pred_df.filter(pl.col('predicted_org') == 'pig')['simpleorg'])
    print('---')


using 1 species.
shape: (2, 2)
┌───────────┬───────┐
│ simpleorg ┆ count │
│ ---       ┆ ---   │
│ str       ┆ u32   │
╞═══════════╪═══════╡
│ pig       ┆ 6131  │
│ human     ┆ 33    │
└───────────┴───────┘
observations matrix shape is: (6164, 1)
classifier accuracy with 1 is: 0.500
11609
observations matrix shape is: (11609, 1)
new_pred_df: shape: (11_609, 2)
┌───────────┬───────────┐
│ acc_index ┆ org_index │
│ ---       ┆ ---       │
│ i64       ┆ i64       │
╞═══════════╪═══════════╡
│ 0         ┆ 0         │
│ 1         ┆ 0         │
│ 2         ┆ 0         │
│ 3         ┆ 0         │
│ 4         ┆ 0         │
│ …         ┆ …         │
│ 11604     ┆ 0         │
│ 11605     ┆ 0         │
│ 11606     ┆ 0         │
│ 11607     ┆ 0         │
│ 11608     ┆ 0         │
└───────────┴───────────┘
shape: (11_609, 2)
┌───────────┬───────────────┐
│ acc_index ┆ predicted_org │
│ ---       ┆ ---           │
│ i64       ┆ str           │
╞═══════════╪═══════════════╡
│ 0         ┆ pig         

In [33]:
print(new_pred_df.filter(pl.col('predicted_org') == 'pig')['simpleorg'].value_counts())

shape: (3, 2)
┌───────────┬───────┐
│ simpleorg ┆ count │
│ ---       ┆ ---   │
│ str       ┆ u32   │
╞═══════════╪═══════╡
│ pig       ┆ 6923  │
│ unknown   ┆ 23688 │
│ human     ┆ 823   │
└───────────┴───────┘
